In [1]:
import numpy as np
import pandas as pd

Goal: To expand the data set columns so that for each article each co-author gets their own row

In [2]:
df = pd.read_csv(r"G:\My Drive\BrainStation\Capstone\Data\2022-02-27 Export\2023-02-27-Cleaned_V1.csv")

In [3]:
df.head()

,Unnamed: 0,level_0_author,total_citations,h_index,article_title,author_info,journal,num_citations,publication_year
0,0,Edward Sargent,118409.0,175.0,Low trap-state density and long carrier diffus...,"Dong Shi, Valerio Adinolfi, Riccardo Comin, Mi...",Science,4165.0,2015.0
1,1,Giles Eperon,50912.0,58.0,Electron-hole diffusion lengths exceeding 1 mi...,"Samuel D Stranks, Giles E Eperon, Giulia Granc...",Science,9177.0,2013.0
2,2,Edward Sargent,118409.0,175.0,Perovskite light-emitting diodes with external...,"Kebin Lin, Jun Xing, Li Na Quan, F Pelayo Garc...",Nature,2299.0,2018.0
3,3,Antonio Abate,44635.0,78.0,Cesium-containing Triple Cation Perovskite Sol...,"Michael Saliba, Taisuke Matsui, Ji-Youn Seo, K...",Energy & Environmental Science,NaN,2016.0
4,4,Giles Eperon,50912.0,58.0,Formamidinium lead trihalide: a broadly tunabl...,"Giles E Eperon, Samuel D Stranks, Christopher ...",Energy & Environmental Science,3649.0,2014.0


In [4]:
def author_split(row):
    # replace and with a commna, rsplit string first, then strip white space, remove any '' elements
    row_list = row.replace('and ', ',').rsplit(',')
    author_list = [i.strip() for i in row_list if i.strip() != '']
    return author_list

Testing `author_split`

In [5]:
test = 'Kelly, Austin, Joyce and Aidan'
test1 = 'Kelly, Austin, Joyce, and Aidan'

In [6]:
author_split(test1)

['Kelly', 'Austin', 'Joyce', 'Aidan']

In [7]:
df.apply({'author_info': author_split}).values[0]

array([list(['Dong Shi', 'Valerio Adinolfi', 'Riccardo Comin', 'Mingjian Yuan', 'Erkki Alarousu', 'Andrei Buin', 'Yin Chen', 'Sjoerd Hoogland', 'Alexander Rothenberger', 'Khabiboulakh Katsiev', 'Yaroslav Losovyj', 'Xin Zhang', 'Peter A Dowben', 'Omar F Mohammed', 'Edward H Sargent', 'Osman M Bakr'])],
      dtype=object)

In [8]:
len(df.apply({'author_info': author_split}).values) == df.shape[0]

True

Write a function that expands the list of co-authors into individual columns.

In [9]:
def arr2df(arr):
    '''
    Parameters:
    ----------
    arr: An array of co-authors names to be expanded
    
    Return:
    ------
    ret: A dataframe of expanded author names
    
    '''
    # Create a DataFrame from the array
    df1 = pd.DataFrame({'author_info': arr})

    # Use the apply(pd.Series) method to expand the list into columns
    df_expanded = df1['author_info'].apply(pd.Series)

    # Rename the columns
    df_expanded.columns = [f"au_{i}" for i in range(df_expanded.shape[1])]

    # Concatenate the original DataFrame and the expanded one
    result = pd.concat([df1, df_expanded], axis=1)

    # Drop the original column
    result = result.drop('author_info', axis=1)
    
    return result

Use the function `arr2df` to create a data frame for all the co-authors

In [10]:
from tqdm import tqdm

auth_arr = df.apply({'author_info': author_split}).values
co_author = arr2df(auth_arr[0])

progress_bar = tqdm(auth_arr[1:])

for idx, i in enumerate(auth_arr[1:]):
    co_author = pd.concat([co_author, arr2df(i)], axis = 0, ignore_index = True)
    progress_bar.set_description(f"Processing {idx}")

Processing 22525:   0%|                                                                      | 0/22526 [13:52<?, ?it/s]

OSError: Cannot save file into a non-existent directory: 'r"G:\My Drive\BrainStation\Capstone'

In [16]:
co_author.to_csv(r"G:\My Drive\BrainStation\Capstone\Data\2022-02-27 Export\2023-02-27-co_author_row.csv")

In [17]:
co_author.shape[0] == df.shape[0]

True

Appending the `co-author` columns to the original data frame.

In [25]:
co_author_full = pd.concat([df, co_author], axis = 1).drop('author_info', axis = 1)

In [26]:
co_author_full.head()

,Unnamed: 0,level_0_author,total_citations,h_index,article_title,journal,num_citations,publication_year,au_0,au_1,...,au_140,au_141,au_142,au_143,au_144,au_145,au_146,au_147,au_148,au_149
0,0,Edward Sargent,118409.0,175.0,Low trap-state density and long carrier diffus...,Science,4165.0,2015.0,Dong Shi,Valerio Adinolfi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Giles Eperon,50912.0,58.0,Electron-hole diffusion lengths exceeding 1 mi...,Science,9177.0,2013.0,Samuel D Stranks,Giles E Eperon,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,Edward Sargent,118409.0,175.0,Perovskite light-emitting diodes with external...,Nature,2299.0,2018.0,Kebin Lin,Jun Xing,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,Antonio Abate,44635.0,78.0,Cesium-containing Triple Cation Perovskite Sol...,Energy & Environmental Science,NaN,2016.0,Michael Saliba,Taisuke Matsui,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,Giles Eperon,50912.0,58.0,Formamidinium lead trihalide: a broadly tunabl...,Energy & Environmental Science,3649.0,2014.0,Giles E Eperon,Samuel D Stranks,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Save to a `.csv` for further processing

In [27]:
co_author_full.to_csv(r"G:\My Drive\BrainStation\Capstone\Data\2022-02-27 Export\2023-02-27-co_author_row_full.csv")